<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="./images/btp-banner.gif" alt="BTP A&C">
</div>

## Tabular Prediction with SAP-RPT-1 and the Product Catalog

In this demo, we will explore how to use **SAP-RPT-1** (Relational Prediction Transformer) to predict missing values in tabular data. Using the same product catalog from the RAG demo, we will ask RPT-1 to predict the **product category** of products based on their attributes — without any model training or fine-tuning.

## Learning Objectives
By the end of this demo, you will be able to:
- Understand what SAP-RPT-1 is and how it differs from LLMs.
- Set up the SAP AI Core client and use the native `RPTClient` from the Generative AI Hub SDK.
- Prepare tabular data in the RPT-1 prediction format using pydantic models.
- Interpret RPT-1 predictions and confidence scores.
- Apply RPT-1 to a real product catalog dataset.

## Requirements

Before starting the Jupyter Notebook steps, ensure the following:
- SAP AI Core instance (extended plan)
- SAP-RPT-1 model available in your AI Core instance (no deployment needed — `RPTClient` resolves the model by name)
- `env_config.json` with your AI Core credentials (same file used in the RAG notebook)

## What is SAP-RPT-1?

**SAP-RPT-1** (Relational Prediction Transformer 1) is a **Relational Foundation Model (RFM)** for tabular and relational data, developed and maintained by SAP.

Unlike Large Language Models (LLMs) that work with text, RPT-1 is designed to work directly with structured, tabular data — the kind found in spreadsheets, databases, and ERP systems.

### Key Characteristics

| | LLM (e.g. Claude) | SAP-RPT-1 |
|---|---|---|
| **Input** | Natural language text | Tabular rows (structured data) |
| **Output** | Text response | Predicted column values + confidence scores |
| **Use case** | Q&A, summarization, generation | Classification, regression on tabular data |
| **Training needed** | No (zero-shot) | No (zero-shot) |
| **Context** | Prompt text | Surrounding rows provide in-context learning |
| **SDK entry point** | `OrchestrationService` / `RPTClient` | `RPTClient` (native SDK) |

### How RPT-1 Works

RPT-1 uses **in-context learning** on tabular data:
1. You provide a table with some rows that have known values for the target column.
2. You mark the rows you want predicted with a special placeholder: **`[PREDICT]`**.
3. RPT-1 learns the pattern from the known rows and predicts the missing values.
4. For classification tasks, RPT-1 also returns a **confidence score** (0-1); for regression tasks the confidence is `null`.

> No model training, no fine-tuning, no embeddings — just send your table and get predictions back.

### Available Models

| Model | Description |
|---|---|
| `sap-rpt-1-large` | Higher accuracy, slower |
| `sap-rpt-1-small` | Faster, lower resource usage |

## About the Data

We use the same product catalog of IT accessory products from the RAG demo. The key attributes used for prediction:

|Field|Description|
|---|---|
|**PRODUCT_ID**|Unique identifier for each product|
|**PRODUCT_NAME**|Product name including brand and model|
|**UNIT_PRICE**|Price in Euros|
|**MANUFACTURER**|Manufacturer name|
|**SUPPLIER_CITY**|City where the supplier is located|
|**STOCK_QUANTITY**|Current stock quantity|
|**LEAD_TIME_DAYS**|Days from order to delivery|
|**MIN_ORDER**|Minimum order quantity|
|**RATING**|Product rating on a scale from 1 to 5|
|**PRODUCT_CATEGORY**|Business product category (Gaming Peripherals, Storage & Memory, Computing, Audio, etc.) — **classification target** ✅|

We will demonstrate one prediction task:
- **Classification**: predict `PRODUCT_CATEGORY` (categorical) — returns confidence scores ✅

> RPT-1 also supports **multi-target prediction** (multiple columns in a single call) and **regression** (numeric targets without confidence scores). These are not shown here but follow the same pattern.

## Setup and Configuration

The following Python modules are used in this notebook.

#### **sap-ai-sdk-gen**
Provides the `RPTClient` for calling SAP-RPT-1 models natively, as well as `AICoreV2Client` and `GenAIHubProxyClient` for authentication.

For more information, please see https://pypi.org/project/sap-ai-sdk-gen/

#### **pandas**
Used to load and display the product catalog in tabular format.

For more information, please see https://pypi.org/project/pandas/

#### Install Python packages

In [ ]:
%pip install "sap-ai-sdk-gen[all]" --break-system-packages -U
%pip install requests pandas --break-system-packages -U

# kernel restart required!!!

#### Restart Python kernel

The Python kernel needs to be restarted before continuing.

> ![title](./images/config_001.png)

#### Configure SAP AI Core credentials

We load the AI Core credentials from `env_config.json` — the same file used in the RAG notebook.

> Please ensure that the Python kernel was restarted!

In [ ]:
import os
import json
import pandas as pd

with open(os.path.join(os.getcwd(), 'env_config.json')) as f:
    aicore_config = json.load(f)

print("✅ Configuration loaded successfully!")
print(f"   AI Core base URL : {aicore_config['AICORE_BASE_URL']}")
print(f"   Resource group   : {aicore_config['AICORE_RESOURCE_GROUP']}")

### Initialize the AI Core Client

We set up the `AICoreV2Client` and `GenAIHubProxyClient` using the same pattern as the RAG notebook. The `RPTClient` uses this proxy client for authentication.

In [ ]:
from ai_core_sdk.ai_core_v2_client import AICoreV2Client
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient

# Set up the AICoreV2Client
ai_core_client = AICoreV2Client(
    base_url=aicore_config['AICORE_BASE_URL'],
    auth_url=aicore_config['AICORE_AUTH_URL'],
    client_id=aicore_config['AICORE_CLIENT_ID'],
    client_secret=aicore_config['AICORE_CLIENT_SECRET'],
    resource_group=aicore_config['AICORE_RESOURCE_GROUP']
)
# Set up the GenAIHubProxyClient
proxy_client = GenAIHubProxyClient(ai_core_client=ai_core_client)
print("✅ AI Core Client connection is established successfully!")

### Initialize the RPT Client

The `RPTClient` is the native SDK client for SAP-RPT-1 models. It handles authentication automatically via the `GenAIHubProxyClient` set up above.

Unlike LLMs which use `init_llm`, RPT-1 uses `RPTClient` because it has a completely different API format — it works with tabular rows and `[PREDICT]` placeholders rather than text prompts.

In [ ]:
from gen_ai_hub.proxy.native.sap import RPTClient, RPTRequest, PredictionConfig, TargetColumn

# Initialize the RPT client (uses the proxy_client set up above)
rpt_client = RPTClient(proxy_client=proxy_client)

# Model to use: sap-rpt-1-large (higher accuracy) or sap-rpt-1-small (faster)
RPT_MODEL = "sap-rpt-1-large"

print("✅ RPT Client initialized successfully!")
print(f"   Model: {RPT_MODEL}")

## Load the Product Catalog

We load the product catalog CSV and select a representative 20-row sample. The first 17 rows will serve as **in-context examples** (known values), and the last 3 will be the rows RPT-1 predicts.

In [ ]:
def parse_price(value) -> float:
    """Parse price string handling German locale variants.

    Handles:
      34.75       → 34.75   (already numeric — returned as-is)
      '34,75'     → 34.75   (comma as decimal separator)
      '2.499,99'  → 2499.99 (dot=thousands, comma=decimal)
      '2,499,99'  → 2499.99 (comma used as both — inconsistent CSV data)
    """
    if isinstance(value, (int, float)):
        return float(value)

    s = str(value).strip()
    comma_count = s.count(',')

    if comma_count > 1:
        # e.g. '2,499,99' — last comma is decimal, preceding commas are thousands
        last_comma = s.rfind(',')
        s = s[:last_comma].replace(',', '') + '.' + s[last_comma + 1:]
    else:
        # e.g. '2.499,99' or '34,75'
        s = s.replace('.', '').replace(',', '.')

    return float(s)


# Load the full product catalog
df = pd.read_csv('data/product_catalog.csv', sep=';', encoding='utf-8-sig')

print(f"Total products in catalog: {len(df)}")
print(f"Columns: {list(df.columns)}")
print()

# Select the columns relevant for RPT-1 prediction
FEATURE_COLUMNS = ['PRODUCT_ID', 'PRODUCT_NAME', 'UNIT_PRICE',
                   'MANUFACTURER', 'SUPPLIER_CITY', 'STOCK_QUANTITY',
                   'LEAD_TIME_DAYS', 'MIN_ORDER', 'RATING', 'PRODUCT_CATEGORY']

# Use a random sample for better representativeness across product categories, manufacturers and rating values
df_sample = df[FEATURE_COLUMNS].sample(n=20, random_state=42).reset_index(drop=True)

# Parse UNIT_PRICE from German locale string to float for clean display and calculations
df_sample['UNIT_PRICE'] = df_sample['UNIT_PRICE'].apply(parse_price)

print("Sample products (20 random rows):")
df_sample

## Prepare the RPT-1 Prediction Request

RPT-1 expects the data in a specific format using pydantic models:
- **`RPTRequest`**: the top-level request object
- **`PredictionConfig`**: specifies which column(s) to predict
- **`TargetColumn`**: defines the target column name and task type (`"regression"` or `"classification"`)
- **`rows`**: list of dicts — known rows have real values, rows to predict use `"[PREDICT]"` as the placeholder

We use the first **17 products** as in-context examples and predict the **last 3**.

In [ ]:
PREDICT_PLACEHOLDER = "[PREDICT]"
INDEX_COLUMN = "PRODUCT_ID"
N_KNOWN = 17  # rows used as in-context examples

predict_ids = [str(df_sample.loc[i, INDEX_COLUMN]) for i in range(N_KNOWN, len(df_sample))]

print(f"Total rows        : {len(df_sample)}")
print(f"Known examples    : {N_KNOWN}")
print(f"Rows to predict   : {len(predict_ids)}")
print(f"Product IDs to predict: {predict_ids}")

## 🏷️ Classification: Predict PRODUCT_CATEGORY

**Task type:** `classification` — predicts a **categorical value** (product category)
**Confidence:** ✅ returned as a score between 0 and 1

We ask RPT-1 to predict the `PRODUCT_CATEGORY` for the last 3 products based on their name, manufacturer, price and other attributes.
The category is a meaningful business grouping (e.g. Gaming Peripherals, Storage & Memory, Computing) — RPT-1 can learn this pattern clearly from the 17 known examples.
Classification tasks return a **confidence score** alongside the predicted value.

In [ ]:
rows_cat = []

for i, row in df_sample.iterrows():
    record = {
        INDEX_COLUMN: str(row[INDEX_COLUMN]),
        "PRODUCT_NAME": str(row["PRODUCT_NAME"]),
        "UNIT_PRICE": parse_price(row["UNIT_PRICE"]),
        "MANUFACTURER": str(row["MANUFACTURER"]),
        "SUPPLIER_CITY": str(row["SUPPLIER_CITY"]),
        "STOCK_QUANTITY": int(row["STOCK_QUANTITY"]) if pd.notna(row["STOCK_QUANTITY"]) else 0,
        "LEAD_TIME_DAYS": int(row["LEAD_TIME_DAYS"]) if pd.notna(row["LEAD_TIME_DAYS"]) else 0,
        "MIN_ORDER": int(row["MIN_ORDER"]) if pd.notna(row["MIN_ORDER"]) else 0,
        "RATING": float(str(row["RATING"]).replace(',', '.')) if pd.notna(row["RATING"]) else 0,
    }
    if i < N_KNOWN:
        record["PRODUCT_CATEGORY"] = str(row["PRODUCT_CATEGORY"])
    else:
        record["PRODUCT_CATEGORY"] = PREDICT_PLACEHOLDER
    rows_cat.append(record)

rpt_request_cat = RPTRequest(
    prediction_config=PredictionConfig(
        target_columns=[
            TargetColumn(name="PRODUCT_CATEGORY", task_type="classification")
        ]
    ),
    rows=rows_cat
)

print(f"Sending classification request to {RPT_MODEL}...")
response_cat = rpt_client.predict(body=rpt_request_cat, model_name=RPT_MODEL)
print("✅ Classification prediction successful!")
print()

# Display predictions with actual vs predicted category and confidence
cat_summary = []
for idx, pred in enumerate(response_cat.predictions):
    pred_dict = pred if isinstance(pred, dict) else pred.model_dump()
    product_id = predict_ids[idx]  # match by position
    cat_preds = pred_dict.get("PRODUCT_CATEGORY", [])
    if cat_preds:
        top = cat_preds[0] if isinstance(cat_preds[0], dict) else cat_preds[0].model_dump()
        actual = df_sample.loc[df_sample['PRODUCT_ID'].astype(str) == product_id, 'PRODUCT_CATEGORY'].values
        cat_summary.append({
            'PRODUCT_ID': product_id,
            'PRODUCT_NAME': df_sample.loc[df_sample['PRODUCT_ID'].astype(str) == product_id, 'PRODUCT_NAME'].values[0],
            'ACTUAL_CATEGORY': actual[0] if len(actual) > 0 else 'N/A',
            'PREDICTED_CATEGORY': top.get('prediction'),
            'CONFIDENCE': round(top.get('confidence'), 3) if top.get('confidence') is not None else None,
            'CORRECT': '✅' if actual[0] == top.get('prediction') else '❌',
        })

df_cat = pd.DataFrame(cat_summary)
print("Classification predictions (PRODUCT_CATEGORY):")
df_cat

> Classification predictions include a **confidence score** (0–1).
> A high confidence means RPT-1 is certain about the predicted category.
> The `CORRECT` column shows whether the prediction matches the actual value.

## Summary

In this notebook, we successfully used **SAP-RPT-1** to predict missing values in the product catalog:

1. **Data Loading**: Loaded the product catalog CSV using pandas.
2. **Client Setup**: Initialized `AICoreV2Client`, `GenAIHubProxyClient`, and `RPTClient` from the SAP AI SDK.
3. **Classification**: Predicted `PRODUCT_CATEGORY` (categorical) — with confidence scores ✅

### Key Takeaways

- **No training required**: RPT-1 uses in-context learning.
- **Native SDK support**: Use `RPTClient` from `gen_ai_hub.proxy.native.sap`.
- **task_type matters**: Use `regression` for numeric targets, `classification` for categorical.
- **Confidence scores**: Returned for **classification** tasks; `null` for regression tasks.
- **Multi-column prediction**: Predict multiple target columns in a single call.
- **Confidence is per-column**: The same request returns `null` for regression and a score for classification.

### RPT-1 vs RAG

| Scenario | RAG (Notebook 1) | RPT-1 (This Notebook) |
|---|---|---|
| Answer questions about products | Yes | No |
| Predict missing product attributes | No | Yes |
| Fill gaps in structured data | No | Yes |
| Summarize product descriptions | Yes | No |